In [21]:
import numpy as np
import pandas as pd
from collections import defaultdict
from scipy.stats import entropy, iqr

In [36]:
df = pd.read_csv("../data/ble_data_labeled_cleaned.csv")
df.head()

,user_id,timestamp,mac_address,RSSI,power,location
0,90,2023-04-10 14:21:46.003,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
1,90,2023-04-10 14:21:46.008,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
2,90,2023-04-10 14:21:46.012,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
3,90,2023-04-10 14:21:46.018,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen
4,90,2023-04-10 14:21:46.024,C9:17:55:E2:3E:0E,-90.0,-2147483648,kitchen


In [37]:
# Map beacon MACs to integer IDs
df = df.sort_values("timestamp").reset_index(drop=True)

# Ensure datetime
if not np.issubdtype(df["timestamp"].dtype, np.datetime64):
    df["timestamp"] = pd.to_datetime(df["timestamp"])

# Map beacon MACs → int IDs (once)
beacon_to_id = {
    mac: i for i, mac in enumerate(df["mac_address"].unique())
}
df["beacon_id"] = df["mac_address"].map(beacon_to_id).astype(np.int16)

# Convert to NumPy arrays
timestamps = df["timestamp"].to_numpy()
rssi       = df["RSSI"].to_numpy()
beacon_ids = df["beacon_id"].to_numpy()
rooms      = df["location"].to_numpy()   # label

In [38]:
def window_ble_data(timestamps, window_size_s=10.0, step_size_s=5.0):
    window_ns = np.timedelta64(int(window_size_s * 1e9), "ns")
    step_ns   = np.timedelta64(int(step_size_s * 1e9), "ns")

    start_time = timestamps[0]
    end_time   = timestamps[-1]

    start_idx = 0
    end_idx   = 0

    t = start_time

    while t + window_ns <= end_time:
        # advance start index
        while start_idx < len(timestamps) and timestamps[start_idx] < t:
            start_idx += 1

        # advance end index
        while end_idx < len(timestamps) and timestamps[end_idx] < t + window_ns:
            end_idx += 1

        if end_idx > start_idx:
            yield start_idx, end_idx

        t += step_ns

In [39]:
def normalize_rssi_per_window(window_df):
    rssi = window_df["RSSI"].values
    if len(rssi) < 2:
        window_df["rssi_norm"] = 0.0
        return window_df

    window_df["rssi_norm"] = (
        rssi - np.mean(rssi)
    ) / (np.std(rssi) + 1e-6)

    return window_df

In [40]:
def extract_fast_features(
    rssi_window,
    beacon_id_window,
    top_k=3
):
    features = {}

    # ---- Aggregate RSSI per beacon (manual, fast) ----
    sums = defaultdict(float)
    counts = defaultdict(int)

    for r, b in zip(rssi_window, beacon_id_window):
        sums[b] += r
        counts[b] += 1

    beacon_ids = np.fromiter(sums.keys(), dtype=np.int16)
    mean_rssi = np.fromiter(
        (sums[b] / counts[b] for b in beacon_ids),
        dtype=np.float32
    )

    # ---- Number of beacons ----
    features["num_beacons_seen"] = len(beacon_ids)

    # ---- Top-K selection (NO full sort) ----
    if len(mean_rssi) > top_k:
        top_idx = np.argpartition(-mean_rssi, top_k)[:top_k]
    else:
        top_idx = np.arange(len(mean_rssi))

    # Sort top-K only
    top_sorted = top_idx[np.argsort(-mean_rssi[top_idx])]

    # ---- Top-K beacon IDs ----
    for i in range(top_k):
        if i < len(top_sorted):
            features[f"rank_{i+1}_beacon"] = beacon_ids[top_sorted[i]]
        else:
            features[f"rank_{i+1}_beacon"] = -1

    # ---- RSSI gaps ----
    if len(top_sorted) > 1:
        features["gap_1_2"] = (
            mean_rssi[top_sorted[0]] - mean_rssi[top_sorted[1]]
        )
    else:
        features["gap_1_2"] = 0.0

    # ---- Packet count strongest ----
    strongest = beacon_ids[top_sorted[0]]
    features["packets_strongest"] = counts[strongest]

    # ---- Cross-beacon dispersion ----
    features["std_rssi_across_beacons"] = np.std(mean_rssi)
    features["range_rssi_across_beacons"] = np.ptp(mean_rssi)

    # ---- Entropy ----
    if len(mean_rssi) > 1:
        p = np.abs(mean_rssi)
        p = p / p.sum()
        features["rssi_entropy"] = entropy(p)
    else:
        features["rssi_entropy"] = 0.0

    return features

In [41]:
rows = []

for i_start, i_end in window_ble_data(
    timestamps,
    window_size_s=10.0,
    step_size_s=5.0
):
    # Extract features
    feats = extract_fast_features(
        rssi[i_start:i_end],
        beacon_ids[i_start:i_end]
    )

    # Label = majority room in the window
    window_rooms = rooms[i_start:i_end]
    room_label = pd.Series(window_rooms).mode()

    if room_label.empty:
        continue

    feats["location"] = room_label.iloc[0]

    rows.append(feats)

features_df = pd.DataFrame(rows)

In [42]:
features_df.to_csv('../data/test2.csv', index=False)